In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import StratifiedKFold


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "src" / "feature_engineering.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Не найден корень проекта. Текущая папка: {CURRENT_DIR}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import (
    prepare_features,
    add_title_hierarchy_features,
)


PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

TARGET_COLUMN = "Цена"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_raw = train.drop(columns=[TARGET_COLUMN]).copy()
y = train[TARGET_COLUMN].copy()

X_features = add_title_hierarchy_features(
    prepare_features(X_raw)
)

In [2]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

# v7 = v6 без «Цвет».
EXCLUDED_COLUMNS_V7 = EXCLUDED_COLUMNS + [
    "Полное название",
    "Цвет",
]

feature_columns_v7 = [
    column
    for column in X_features.columns
    if column not in EXCLUDED_COLUMNS_V7
]

X_model_v7 = X_features[
    feature_columns_v7
].copy()

numeric_columns_v7 = X_model_v7.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_columns_v7 = [
    column
    for column in feature_columns_v7
    if column not in numeric_columns_v7
]

for column in numeric_columns_v7:
    X_model_v7[column] = pd.to_numeric(
        X_model_v7[column],
        errors="coerce",
    ).astype(float)

for column in categorical_columns_v7:
    X_model_v7[column] = (
        X_model_v7[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

assert "Полное название" not in X_model_v7.columns
assert "Цвет" not in X_model_v7.columns
assert X_model_v7.shape[1] == 57

print("X_model_v7:", X_model_v7.shape)
print("Categorical:", len(categorical_columns_v7))

X_model_v7: (8340, 57)
Categorical: 17


In [3]:
def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


cv_bins = np.asarray(
    pd.qcut(
        y,
        q=10,
        labels=False,
        duplicates="drop",
    )
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

oof_v7_predictions = np.zeros(
    len(train),
    dtype=float,
)

v7_fold_rows = []

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X_model_v7, cv_bins),
    start=1,
):
    print(f"\n{'=' * 70}")
    print(f"Fold {fold}/5")
    print(f"{'=' * 70}")

    X_train_fold = X_model_v7.iloc[train_idx]
    X_valid_fold = X_model_v7.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    model_v7 = CatBoostRegressor(
        loss_function="RMSE",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_seed=42,
        allow_writing_files=False,
        verbose=500,
    )

    model_v7.fit(
        X_train_fold,
        np.log1p(y_train_fold),
        cat_features=categorical_columns_v7,
        eval_set=(
            X_valid_fold,
            np.log1p(y_valid_fold),
        ),
        use_best_model=True,
        early_stopping_rounds=200,
    )

    valid_pred = np.maximum(
        np.expm1(
            model_v7.predict(X_valid_fold)
        ),
        1,
    )

    oof_v7_predictions[valid_idx] = valid_pred

    v7_fold_rows.append(
        {
            "fold": fold,
            "best_iteration": model_v7.get_best_iteration(),
            "fold_mape_pct": mape_percent(
                y_valid_fold,
                valid_pred,
            ),
        }
    )

v7_fold_results = pd.DataFrame(v7_fold_rows)

display(v7_fold_results)

print(
    "\nV7 OOF MAPE:",
    mape_percent(y, oof_v7_predictions),
)


Fold 1/5
0:	learn: 0.6532994	test: 0.6460489	best: 0.6460489 (0)	total: 235ms	remaining: 11m 43s
500:	learn: 0.1511771	test: 0.2132972	best: 0.2132730 (498)	total: 1m 21s	remaining: 6m 46s
1000:	learn: 0.1136363	test: 0.2048234	best: 0.2048111 (996)	total: 2m 56s	remaining: 5m 53s
1500:	learn: 0.0909894	test: 0.2022867	best: 0.2022777 (1498)	total: 3m 53s	remaining: 3m 53s
2000:	learn: 0.0766321	test: 0.2010116	best: 0.2010116 (2000)	total: 4m 39s	remaining: 2m 19s
2500:	learn: 0.0648097	test: 0.2000437	best: 0.2000325 (2495)	total: 5m 24s	remaining: 1m 4s
2999:	learn: 0.0557876	test: 0.1997206	best: 0.1996933 (2982)	total: 6m 9s	remaining: 0us

bestTest = 0.1996932988
bestIteration = 2982

Shrink model to first 2983 iterations.

Fold 2/5
0:	learn: 0.6518579	test: 0.6574329	best: 0.6574329 (0)	total: 84ms	remaining: 4m 11s
500:	learn: 0.1585236	test: 0.1919387	best: 0.1919387 (500)	total: 41.5s	remaining: 3m 26s
1000:	learn: 0.1207752	test: 0.1806325	best: 0.1806325 (1000)	total: 1m 2

,fold,best_iteration,fold_mape_pct
0,1,2982,13.534112
1,2,2998,12.116378
2,3,2991,12.645774
3,4,2997,12.957613
4,5,2999,12.154349



V7 OOF MAPE: 12.681645281534081


In [4]:
v7_oof_path = (
    REPORTS_DIR
    / "catboost_title_hierarchy_v7_no_color_oof.parquet"
)

v7_oof = pd.DataFrame(
    {
        "car_id": train["car_id"].to_numpy(),
        "y_true": y.to_numpy(),
        "title_v7_pred": oof_v7_predictions,
        "ape_pct": (
            np.abs(
                oof_v7_predictions - y.to_numpy()
            )
            / y.to_numpy()
            * 100
        ),
    }
)

v7_oof.to_parquet(
    v7_oof_path,
    index=False,
)

print("Saved:", v7_oof_path)

Saved: C:\temp\shift_ml\reports\catboost_title_hierarchy_v7_no_color_oof.parquet


In [5]:
v6_oof = pd.read_parquet(
    REPORTS_DIR
    / "catboost_title_hierarchy_v6_oof_clean.parquet"
)

assert np.array_equal(
    v6_oof["car_id"].to_numpy(),
    v7_oof["car_id"].to_numpy(),
)

comparison_v6_v7 = pd.DataFrame(
    {
        "model": ["v6", "v7_no_color"],
        "oof_mape_pct": [
            mape_percent(
                v6_oof["y_true"],
                v6_oof["title_v6_pred"],
            ),
            mape_percent(
                v7_oof["y_true"],
                v7_oof["title_v7_pred"],
            ),
        ],
    }
)

display(comparison_v6_v7)

print(
    "Mean abs difference v6 vs v7:",
    np.mean(
        np.abs(
            v6_oof["title_v6_pred"].to_numpy()
            - v7_oof["title_v7_pred"].to_numpy()
        )
    ),
)

print(
    "Prediction correlation:",
    np.corrcoef(
        v6_oof["title_v6_pred"],
        v7_oof["title_v7_pred"],
    )[0, 1],
)

,model,oof_mape_pct
0,v6,12.746942
1,v7_no_color,12.681645


Mean abs difference v6 vs v7: 1247.3327785425856
Prediction correlation: 0.9959305772846304


In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_percentage_error


def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


REPORTS_DIR = PROJECT_ROOT / "reports"

ridge_oof = pd.read_parquet(
    REPORTS_DIR / "ridge_oof_predictions_alpha_0_1.parquet"
)

v5_oof = pd.read_parquet(
    REPORTS_DIR / "catboost_title_hierarchy_v5_oof_fresh.parquet"
)

v6_oof = pd.read_parquet(
    REPORTS_DIR / "catboost_title_hierarchy_v6_oof_clean.parquet"
)

v7_oof = pd.read_parquet(
    REPORTS_DIR / "catboost_title_hierarchy_v7_no_color_oof.parquet"
)

text_oof = pd.read_parquet(
    REPORTS_DIR / "text_ridge_tfidf_oof_predictions.parquet"
)

for name, df in {
    "ridge": ridge_oof,
    "v5": v5_oof,
    "v6": v6_oof,
    "v7": v7_oof,
    "text": text_oof,
}.items():
    print(name, df.columns.tolist(), df.shape)

ridge ['car_id', 'y_true', 'ridge_pred'] (8340, 3)
v5 ['car_id', 'y_true', 'title_v5_pred'] (8340, 3)
v6 ['car_id', 'y_true', 'title_v6_pred'] (8340, 3)
v7 ['car_id', 'y_true', 'title_v7_pred', 'ape_pct'] (8340, 4)
text ['car_id', 'y_true', 'text_ridge_pred', 'ape_pct'] (8340, 4)


In [7]:
reference_car_id = v7_oof["car_id"].to_numpy()

for name, df in {
    "ridge": ridge_oof,
    "v5": v5_oof,
    "v6": v6_oof,
    "text": text_oof,
}.items():
    assert np.array_equal(
        df["car_id"].to_numpy(),
        reference_car_id,
    ), f"Порядок car_id не совпал: {name}"

assert np.array_equal(
    v7_oof["y_true"].to_numpy(),
    v6_oof["y_true"].to_numpy(),
)

ensemble_frame = pd.DataFrame(
    {
        "car_id": reference_car_id,
        "y_true": v7_oof["y_true"].to_numpy(),
        "ridge": ridge_oof["ridge_pred"].to_numpy(),
        "v5": v5_oof["title_v5_pred"].to_numpy(),
        "v6": v6_oof["title_v6_pred"].to_numpy(),
        "v7": v7_oof["title_v7_pred"].to_numpy(),
        "text": text_oof["text_ridge_pred"].to_numpy(),
    }
)

display(ensemble_frame.head())

,car_id,y_true,ridge,v5,v6,v7,text
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,38812,40566.385555,47403.606067,46820.046301,45060.668121,41370.727136
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,15950,16612.439448,14574.018047,13640.264268,14103.236253,17044.414914
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,41990,40958.198793,41213.010617,40781.144022,40106.871323,41271.250267
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,69900,59670.249223,79948.430623,74796.188103,77854.687362,67829.699195
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,27950,27712.258776,30176.622498,30123.132359,28039.087064,28567.945931


In [8]:
model_columns = [
    "ridge",
    "v5",
    "v6",
    "v7",
    "text",
]

single_model_results = pd.DataFrame(
    {
        "model": model_columns,
        "oof_mape_pct": [
            mape_percent(
                ensemble_frame["y_true"],
                ensemble_frame[column],
            )
            for column in model_columns
        ],
    }
).sort_values(
    "oof_mape_pct"
).reset_index(drop=True)

display(single_model_results)

display(
    ensemble_frame[
        model_columns
    ].corr().round(5)
)

,model,oof_mape_pct
0,v7,12.681645
1,v6,12.746942
2,v5,12.755321
3,ridge,14.468951
4,text,14.903406


,ridge,v5,v6,v7,text
ridge,1.00000,0.85669,0.85801,0.86277,0.88813
v5,0.85669,1.00000,0.99644,0.99571,0.93253
v6,0.85801,0.99644,1.00000,0.99593,0.93338
v7,0.86277,0.99571,0.99593,1.00000,0.93460
text,0.88813,0.93253,0.93338,0.93460,1.00000


In [9]:
y_oof = ensemble_frame["y_true"].to_numpy()

prediction_matrix = ensemble_frame[
    model_columns
].to_numpy()


def objective(weights: np.ndarray) -> float:
    blended_pred = prediction_matrix @ weights

    return mape_percent(
        y_oof,
        np.maximum(blended_pred, 1),
    )


constraints = [
    {
        "type": "eq",
        "fun": lambda weights: weights.sum() - 1,
    }
]

bounds = [
    (0, 1)
    for _ in model_columns
]

starting_weights = [
    np.full(
        len(model_columns),
        1 / len(model_columns),
    ),
    np.array([0.25, 0.32, 0.33, 0.00, 0.10]),
    np.array([0.25, 0.32, 0.00, 0.33, 0.10]),
    np.array([0.30, 0.30, 0.00, 0.30, 0.10]),
]

optimization_rows = []

for initial_weights in starting_weights:
    result = minimize(
        objective,
        x0=initial_weights,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": 10_000,
            "ftol": 1e-12,
        },
    )

    optimization_rows.append(
        {
            "success": result.success,
            "message": result.message,
            "oof_mape_pct": result.fun,
            "weights": result.x,
        }
    )

optimization_results = pd.DataFrame(
    optimization_rows
).sort_values(
    "oof_mape_pct"
).reset_index(drop=True)

display(optimization_results)

,success,message,oof_mape_pct,weights
0,True,Optimization terminated successfully,12.044387,"[0.2554245485479869, 0.194425271488974, 0.1694..."
1,True,Optimization terminated successfully,12.044387,"[0.2554240133011093, 0.1944242360644072, 0.169..."
2,True,Optimization terminated successfully,12.044387,"[0.2554237303456689, 0.19442350635398575, 0.16..."
3,True,Optimization terminated successfully,12.044387,"[0.25542361540271386, 0.19444156573780585, 0.1..."


In [10]:
best_weights = optimization_results.loc[
    0,
    "weights",
]

best_ensemble_pred = prediction_matrix @ best_weights

best_weights_table = pd.DataFrame(
    {
        "model": model_columns,
        "weight": best_weights,
    }
).sort_values(
    "weight",
    ascending=False,
).reset_index(drop=True)

display(best_weights_table)

print(
    "Best five-model raw OOF MAPE:",
    mape_percent(
        y_oof,
        best_ensemble_pred,
    ),
)

,model,weight
0,v7,0.297735
1,ridge,0.255425
2,v5,0.194425
3,v6,0.169445
4,text,0.082970


Best five-model raw OOF MAPE: 12.044386897997068


In [11]:
ensemble_variants = {
    "old_4_models": [
        "ridge",
        "v5",
        "v6",
        "text",
    ],
    "replace_v6_with_v7": [
        "ridge",
        "v5",
        "v7",
        "text",
    ],
    "all_5_models": [
        "ridge",
        "v5",
        "v6",
        "v7",
        "text",
    ],
}

ablation_rows = []

for variant_name, columns in ensemble_variants.items():
    matrix = ensemble_frame[columns].to_numpy()

    def variant_objective(weights):
        return mape_percent(
            y_oof,
            np.maximum(matrix @ weights, 1),
        )

    result = minimize(
        variant_objective,
        x0=np.full(len(columns), 1 / len(columns)),
        method="SLSQP",
        bounds=[(0, 1)] * len(columns),
        constraints=[
            {
                "type": "eq",
                "fun": lambda weights: weights.sum() - 1,
            }
        ],
        options={
            "maxiter": 10_000,
            "ftol": 1e-12,
        },
    )

    ablation_rows.append(
        {
            "variant": variant_name,
            "oof_mape_pct": result.fun,
            "weights": dict(
                zip(
                    columns,
                    result.x.round(6),
                )
            ),
        }
    )

ensemble_ablation_results = (
    pd.DataFrame(ablation_rows)
    .sort_values("oof_mape_pct")
    .reset_index(drop=True)
)

display(ensemble_ablation_results)

,variant,oof_mape_pct,weights
0,all_5_models,12.044387,"{'ridge': 0.255424, 'v5': 0.194424, 'v6': 0.16..."
1,replace_v6_with_v7,12.053082,"{'ridge': 0.255193, 'v5': 0.282194, 'v7': 0.37..."
2,old_4_models,12.083346,"{'ridge': 0.253287, 'v5': 0.321436, 'v6': 0.32..."


v7 добили корректно. Итог:

Старый 4-model ensemble: 12.083346% OOF
Новый 5-model ensemble: 12.044387% OOF
Улучшение:              −0.038959 п.п.

Да, прибавка небольшая, потому что v6 и v7 почти дублируют друг друга. Но оптимизатор оставил оба — значит, у каждого есть немного своей полезной ошибки.

In [12]:
FINAL_WEIGHTS_V2 = {
    "ridge": 0.255425,
    "v5": 0.194425,
    "v6": 0.169445,
    "v7": 0.297735,
    "text": 0.082970,
}

Пока не переобучаем финальные модели и не делаем submission. Калибровку нового ансамбля тоже отложим: сначала попробуем добыть новый сигнал, потом один раз пересоберём всё вместе.

Возвращаемся к текущему сильному направлению — исходные дубли до canonicalization.

Но сначала найдём реальные файлы без предположения, что они лежат именно в data/raw/train.csv:

In [13]:
from pathlib import Path

data_files = sorted(
    path
    for path in (PROJECT_ROOT / "data").rglob("*")
    if path.suffix.lower() in {".csv", ".parquet"}
)

for path in data_files:
    print(path.relative_to(PROJECT_ROOT))

data\processed\train_canonical.parquet
data\processed\X_test_canonical.parquet
data\processed\X_train_canonical.parquet


Дальше запусти диагностику по дублям, заменив пути на найденные:

In [15]:
from pathlib import Path

def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"Unsupported format: {path.suffix}")


table_candidates = []

for path in (PROJECT_ROOT / "data").rglob("*"):
    if path.suffix.lower() not in {".csv", ".parquet"}:
        continue

    try:
        table = read_table(path)
    except Exception as error:
        print(f"Skip: {path.name} | {error}")
        continue

    if "car_id" not in table.columns:
        continue

    table_candidates.append(
        {
            "path": path,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "rows": len(table),
            "columns": len(table.columns),
            "has_target": "Цена" in table.columns,
            "unique_car_id": table["car_id"].nunique(),
        }
    )

candidates_df = (
    pd.DataFrame(table_candidates)
    .sort_values(["has_target", "rows"], ascending=[False, False])
    .reset_index(drop=True)
)

display(
    candidates_df[
        [
            "relative_path",
            "rows",
            "columns",
            "has_target",
            "unique_car_id",
        ]
    ]
)

,relative_path,rows,columns,has_target,unique_car_id
0,data\processed\train_canonical.parquet,8340,23,True,8340
1,data\processed\X_test_canonical.parquet,8341,22,False,8341
2,data\processed\X_train_canonical.parquet,8340,22,False,8340


In [16]:
raw_train_path = (
    candidates_df
    .query("has_target == True")
    .sort_values("rows", ascending=False)
    .iloc[0]["path"]
)

raw_test_path = (
    candidates_df
    .query("has_target == False")
    .sort_values("rows", ascending=False)
    .iloc[0]["path"]
)

raw_train = read_table(raw_train_path)
raw_test = read_table(raw_test_path)

print("Raw train:", raw_train_path.relative_to(PROJECT_ROOT), raw_train.shape)
print("Raw test: ", raw_test_path.relative_to(PROJECT_ROOT), raw_test.shape)

assert raw_train["car_id"].nunique() == 8340
assert raw_test["car_id"].nunique() == 8341
assert len(raw_train) > raw_train["car_id"].nunique()
assert len(raw_test) > raw_test["car_id"].nunique()

Raw train: data\processed\train_canonical.parquet (8340, 23)
Raw test:  data\processed\X_test_canonical.parquet (8341, 22)


AssertionError: 

In [17]:
from pathlib import Path

SEARCH_TERMS = [
    "read_csv",
    "read_parquet",
    "train.csv",
    "test.csv",
    "canonical",
    ".zip",
]

TEXT_EXTENSIONS = {".py", ".ipynb", ".md", ".txt"}

matches = []

for path in PROJECT_ROOT.rglob("*"):
    if (
        not path.is_file()
        or ".venv" in path.parts
        or path.suffix.lower() not in TEXT_EXTENSIONS
    ):
        continue

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )
    except Exception:
        continue

    lines = text.splitlines()

    for line_number, line in enumerate(lines, start=1):
        if any(term.lower() in line.lower() for term in SEARCH_TERMS):
            matches.append(
                {
                    "file": str(path.relative_to(PROJECT_ROOT)),
                    "line": line_number,
                    "text": line.strip()[:300],
                }
            )

matches_df = pd.DataFrame(matches)

display(matches_df.head(200))

,file,line,text
0,notebooks\00_explore_task_structure.ipynb,1209,"""x_test_base = pd.read_csv(DATA_DIR / \""X_test..."
1,notebooks\00_explore_task_structure.ipynb,1210,"""y_train_base = pd.read_csv(DATA_DIR / \""y_tra..."
2,notebooks\00_explore_task_structure.ipynb,2039,""" return pd.read_csv(path)\n"","
3,notebooks\00_explore_task_structure.ipynb,2042,""" return pd.read_parquet(path)\n"","
4,notebooks\00_explore_task_structure.ipynb,3122,"""X_base = pd.read_csv(DATA_DIR / \""X_test_base..."
...,...,...,...
106,train_data\baseline_for_students.ipynb,118,"""X_test = pd.read_csv('train_data/X_test.csv')..."
107,train_data\baseline_for_students.ipynb,679,"""# Архивируем в submission.zip\n"","
108,train_data\baseline_for_students.ipynb,680,"""with zipfile.ZipFile('submission.zip', 'w', z..."
109,train_data\baseline_for_students.ipynb,705,"""# Архивируем в submission.zip\n"","


In [18]:
for path in sorted((PROJECT_ROOT / "train_data").iterdir()):
    if path.is_file():
        print(
            f"{path.name:35} "
            f"{path.stat().st_size / 1024 / 1024:8.2f} MB"
        )

baseline_for_students.ipynb             0.03 MB
car_id_coverage.csv                     0.42 MB
data_files_inventory.csv                0.00 MB
datasets_profile.csv                    0.02 MB
source_audit.csv                        0.00 MB
X_test_base.csv                         1.78 MB
y_train_base.csv                        0.34 MB


In [19]:
X_train_base = pd.read_csv(
    PROJECT_ROOT / "train_data" / "X_train.csv"
)

y_train_base = pd.read_csv(
    PROJECT_ROOT / "train_data" / "y_train.csv"
)

X_test_base = pd.read_csv(
    PROJECT_ROOT / "train_data" / "X_test.csv"
)

print("X_train:", X_train_base.shape)
print("y_train:", y_train_base.shape)
print("X_test: ", X_test_base.shape)

print("\nX_train columns:")
print(X_train_base.columns.tolist())

print("\ny_train columns:")
print(y_train_base.columns.tolist())

print("\nUnique car_id:")
print("X_train:", X_train_base["car_id"].nunique())
print("X_test: ", X_test_base["car_id"].nunique())

print("\nRows per car in X_train:")
display(
    X_train_base.groupby("car_id")
    .size()
    .value_counts()
    .sort_index()
    .rename("cars_count")
    .to_frame()
    .head(25)
)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\temp\\shift_ml\\train_data\\X_train.csv'

In [20]:
from pathlib import Path

all_data_files = []

for path in PROJECT_ROOT.rglob("*"):
    if (
        not path.is_file()
        or ".venv" in path.parts
        or path.suffix.lower() not in {
            ".csv",
            ".parquet",
            ".xlsx",
            ".xls",
            ".json",
            ".zip",
        }
    ):
        continue

    all_data_files.append(
        {
            "path": str(path.relative_to(PROJECT_ROOT)),
            "extension": path.suffix.lower(),
            "size_mb": round(
                path.stat().st_size / 1024 / 1024,
                3,
            ),
        }
    )

all_data_files = (
    pd.DataFrame(all_data_files)
    .sort_values("size_mb", ascending=False)
    .reset_index(drop=True)
)

display(all_data_files)

,path,extension,size_mb
0,train_data\X_test_base.csv,.csv,1.782
1,train_data\dataset_23\data.json,.json,1.634
2,train_data\dataset_19\data.json,.json,1.610
3,train_data\dataset_9\data.json,.json,1.571
4,train_data\dataset_31\data.json,.json,1.564
...,...,...,...
89,train_data\dataset_1\data.csv,.csv,0.038
90,train_data\datasets_profile.csv,.csv,0.024
91,train_data\data_files_inventory.csv,.csv,0.003
92,train_data\source_audit.csv,.csv,0.002


In [21]:
y_train_base = pd.read_csv(
    PROJECT_ROOT / "train_data" / "y_train_base.csv"
)

print(y_train_base.shape)
print(y_train_base.columns.tolist())

display(y_train_base.head())
display(y_train_base.describe(include="all"))

(8340, 2)
['car_id', 'Цена']


,car_id,Цена
0,099ec55b-6322-4a9f-8d5e-0bed830c3023,28777
1,25e3e231-5efc-4032-abe9-794c0cba8295,62999
2,54f3b009-8e37-49cd-905a-9637d6bf135c,28999
3,b361a8ff-af76-40cc-9f54-811b6bfd5eb5,33950
4,671c0cce-667a-4f44-87e7-fe999e17e55e,77888


,car_id,Цена
count,8340,8.340000e+03
unique,8340,NaN
top,099ec55b-6322-4a9f-8d5e-0bed830c3023,NaN
freq,1,NaN
mean,NaN,3.712697e+04
std,NaN,3.679723e+04
min,NaN,9.000000e+02
25%,NaN,1.944075e+04
50%,NaN,2.962000e+04
75%,NaN,4.399000e+04


In [22]:
import json

sample_json_path = (
    PROJECT_ROOT
    / "train_data"
    / "dataset_23"
    / "data.json"
)

with open(sample_json_path, "r", encoding="utf-8") as file:
    sample_json = json.load(file)

print("Type:", type(sample_json))

if isinstance(sample_json, dict):
    print("Top-level keys:", sample_json.keys())

    for key, value in sample_json.items():
        print(
            f"\nKEY: {key}"
            f"\nTYPE: {type(value)}"
        )

        if isinstance(value, list):
            print("List length:", len(value))
            print("First item:")
            print(value[0] if value else None)

        elif isinstance(value, dict):
            print("Nested keys:", value.keys())

        else:
            print("Value:", value)

elif isinstance(sample_json, list):
    print("List length:", len(sample_json))
    print("\nFirst item:")
    print(sample_json[0])

Type: <class 'list'>
List length: 1128

First item:
{'car_id': 'b1b87bbb-0d7d-4b3f-8a06-5e032c37431c', 'Бренд': 'TOYOTA', 'Год выпуска': 2015.0, 'Модель': 'VELLFIRE', 'Тип машины': 'USED DEALER AD', 'Полное название': '2015 TOYOTA VELLFIRE SC-PACKAGE HYBRID 2.5L AWD 7 SEATER', 'Исползование': 'USED', 'КПП': '-', 'Двигатель': '-', 'Привод': 'Other', 'Топливо': '-', 'Расход': '-', 'Пробег': '67704', 'Цвет': 'Black / -', 'Локация': 'MOORABBIN, VIC', 'Количество цилиндров': '-', 'Тип кузова': 'WAGON', 'Двери': None, 'Количество кресел': None, 'Оценка эксперта': 4.0, 'Количество владельцев': 8.0, 'Предложение': '61052'}


In [23]:
json_paths = sorted(
    (PROJECT_ROOT / "train_data").glob(
        "dataset_*/data.json"
    )
)

json_inventory = []

for path in json_paths:
    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    row = {
        "path": str(path.relative_to(PROJECT_ROOT)),
        "top_level_type": type(data).__name__,
    }

    if isinstance(data, list):
        row["n_items"] = len(data)

        if data and isinstance(data[0], dict):
            row["first_item_keys"] = sorted(data[0].keys())
            row["has_car_id"] = "car_id" in data[0]

    elif isinstance(data, dict):
        row["n_items"] = None
        row["top_level_keys"] = sorted(data.keys())
        row["has_car_id"] = "car_id" in data

    json_inventory.append(row)

json_inventory = pd.DataFrame(json_inventory)

display(json_inventory)

,path,top_level_type,n_items,first_item_keys,has_car_id
0,train_data\dataset_17\data.json,list,1057,"[car_id, Бренд, Год выпуска, Двери, Двигатель,...",True
1,train_data\dataset_19\data.json,list,1110,"[car_id, Бренд, Год выпуска, Двери, Двигатель,...",True
2,train_data\dataset_23\data.json,list,1128,"[car_id, Бренд, Год выпуска, Двери, Двигатель,...",True
3,train_data\dataset_31\data.json,list,1079,"[car_id, Бренд, Год выпуска, Двери, Двигатель,...",True
4,train_data\dataset_37\data.json,list,1042,"[car_id, Бренд, Год выпуска, Двери, Двигатель,...",True
5,train_data\dataset_39\data.json,list,1062,"[car_id, Бренд, Год выпуска, Двери, Двигатель,...",True
6,train_data\dataset_9\data.json,list,1084,"[car_id, Бренд, Год выпуска, Двери, Двигатель,...",True
